In [ ]:
# Install all required packages
!pip install transformers datasets torch matplotlib

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch
import matplotlib.pyplot as plt
from collections import Counter
import time

In [ ]:
# 1. Load WikiText-2 dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
print(f"Number of lines in dataset: {len(dataset)}")

Dataset Statistics Before Tokenization


Enhancement: sequence length histogram before chunking

In [ ]:
# ENHANCEMENT: Compute raw text length distribution before tokenizing
text_lengths = [len(example["text"].split()) for example in dataset if example["text"].strip()]

print(f"Total non-empty lines: {len(text_lengths)}")
print(f"Mean word count:       {sum(text_lengths)/len(text_lengths):.1f}")
print(f"Max word count:        {max(text_lengths)}")
print(f"Min word count:        {min(text_lengths)}")

plt.figure(figsize=(10, 4))
plt.hist(text_lengths, bins=80, color="steelblue", edgecolor="white")
plt.title("Distribution of Raw Text Lengths (word count per line)")
plt.xlabel("Words per line")
plt.ylabel("Frequency")
plt.yscale("log")  # log scale because many lines are very short
plt.tight_layout()
plt.show()

Initialize GPT tokenizer

In [ ]:
# 2. Initialize GPT-2 tokenizer
tokenizer_gpt2 = AutoTokenizer.from_pretrained("gpt2")
tokenizer_gpt2.pad_token = tokenizer_gpt2.eos_token

# I used gpt2 as the primary tokenizer going forward
tokenizer = tokenizer_gpt2

Multi-Tokenizer Comparison


Enhancement: compare GPT-2 vs BERT vs DistilBERT on the same sample sentences

In [ ]:
# ENHANCEMENT: Compare tokenizers — GPT-2 vs BERT vs DistilBERT
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer_distilbert = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample_texts = [
    "The history of the Roman Empire spans several centuries.",
    "Tokenization is a fundamental step in NLP pipelines.",
    "Pneumonoultramicroscopicsilicovolcanoconiosis is a lung disease.",
]

print(f"{'Model':<20} {'Vocab Size':<12} {'Tokens (s1)':<14} {'Tokens (s2)':<14} {'Tokens (s3)'}")
print("-" * 75)

for name, tok in [("GPT-2", tokenizer_gpt2),
                   ("BERT", tokenizer_bert),
                   ("DistilBERT", tokenizer_distilbert)]:
    counts = [len(tok.encode(s)) for s in sample_texts]
    print(f"{name:<20} {tok.vocab_size:<12} {counts[0]:<14} {counts[1]:<14} {counts[2]}")

Tokenize Dataset

In [ ]:
# 3. Tokenize the full dataset with batched .map()
def tokenize_function(examples):
    return tokenizer(examples["text"], return_special_tokens_mask=False)

tokenized_ds = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print(tokenized_ds[0]["input_ids"][:20])  # sanity check: first 20 token IDs

Vocabulary Frequency Analysis


Enhancement: token frequency distribution + top/bottom tokens

In [ ]:
# ENHANCEMENT: Vocabulary analysis — token frequency across the dataset
all_token_ids = []
for example in tokenized_ds:
    all_token_ids.extend(example["input_ids"])

token_counts = Counter(all_token_ids)

print(f"Total tokens in dataset:  {len(all_token_ids):,}")
print(f"Unique tokens used:       {len(token_counts):,} / {tokenizer.vocab_size:,}")

# Top 20 most common tokens
top20 = token_counts.most_common(20)
top20_tokens = [tokenizer.decode([tid]) for tid, _ in top20]
top20_counts = [cnt for _, cnt in top20]

# Bottom 20 (rarest used tokens)
bottom20 = token_counts.most_common()[:-21:-1]
bottom20_tokens = [tokenizer.decode([tid]) for tid, _ in bottom20]

print(f"\nTop 20 most frequent tokens:  {list(zip(top20_tokens, top20_counts))}")
print(f"\nBottom 20 rarest tokens:      {[t for t in bottom20_tokens]}")

# Plot top 20
plt.figure(figsize=(12, 4))
plt.bar(range(20), top20_counts, color="coral", edgecolor="white")
plt.xticks(range(20), [repr(t) for t in top20_tokens], rotation=45, ha="right", fontsize=9)
plt.title("Top 20 Most Frequent Tokens (GPT-2 tokenizer, WikiText-2)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

Block Size Comparison


Enhancement: try 3 block sizes and compare sequence counts

In [ ]:
# ENHANCEMENT: Compare block sizes — 128 vs 256 vs 512
def group_texts(examples, block_size):
    concatenated_inputs = sum(examples["input_ids"], [])
    concatenated_masks  = sum(examples["attention_mask"], [])
    total_len = (len(concatenated_inputs) // block_size) * block_size
    result_input_ids = [concatenated_inputs[i:i+block_size] for i in range(0, total_len, block_size)]
    result_masks     = [concatenated_masks[i:i+block_size]  for i in range(0, total_len, block_size)]
    return {"input_ids": result_input_ids, "attention_mask": result_masks}

block_sizes = [128, 256, 512]
block_results = {}

for bs in block_sizes:
    ds = tokenized_ds.map(lambda ex: group_texts(ex, bs), batched=True, batch_size=1000)
    block_results[bs] = len(ds)
    print(f"block_size={bs:>4}  →  {len(ds):>6} sequences")

plt.figure(figsize=(7, 4))
plt.bar([str(b) for b in block_sizes], list(block_results.values()), color="mediumseagreen", edgecolor="white", width=0.4)
plt.title("Number of Sequences by Block Size")
plt.xlabel("Block size (tokens)")
plt.ylabel("Number of sequences")
plt.tight_layout()
plt.show()

Use block_size=128 as Primary (original step 4)

In [ ]:
# 4. Used block_size=128 as the primary dataset for the rest of the lab
block_size = 128
lm_ds = tokenized_ds.map(lambda ex: group_texts(ex, block_size), batched=True, batch_size=1000)
print(f"LM training sequences: {len(lm_ds)}")

Attention Mask Visualization


Enhancement: visualize padding vs real tokens across a batch

In [ ]:
# ENHANCEMENT: Attention mask visualization for first 16 sequences
sample_ids = [lm_ds[i]["attention_mask"] for i in range(16)]
mask_tensor = torch.tensor(sample_ids, dtype=torch.float)  # shape [16, 128]

plt.figure(figsize=(14, 4))
plt.imshow(mask_tensor, aspect="auto", cmap="Blues", interpolation="nearest")
plt.colorbar(label="Mask value (1=real, 0=pad)")
plt.title("Attention Masks for First 16 Sequences\n(white = padding, blue = real token)")
plt.xlabel("Token position")
plt.ylabel("Sequence index")
plt.tight_layout()
plt.show()

# Count how much padding exists across this sample
pad_fraction = (mask_tensor == 0).sum().item() / mask_tensor.numel()
print(f"Padding fraction in this sample: {pad_fraction:.2%}")

DataLoader with Multi-Worker Benchmark


Enhancement: benchmark num_workers=0 vs num_workers=2

In [ ]:
# 5. Build DataLoader — benchmark single vs multi-worker throughput
def collate_fn(batch):
    input_ids = torch.tensor([example["input_ids"] for example in batch], dtype=torch.long)
    return {"input_ids": input_ids, "labels": input_ids.clone()}

timing_results = {}

for num_workers in [0, 2]:
    loader = DataLoader(lm_ds, batch_size=8, shuffle=True,
                        collate_fn=collate_fn, num_workers=num_workers)
    start = time.time()
    for i, batch in enumerate(loader):
        if i == 50:  # time 50 batches
            break
    elapsed = time.time() - start
    timing_results[num_workers] = elapsed
    print(f"num_workers={num_workers}  →  50 batches in {elapsed:.2f}s  ({50/elapsed:.1f} batches/sec)")

# Keep num_workers=2 loader as primary
train_loader = DataLoader(lm_ds, batch_size=8, shuffle=True,
                          collate_fn=collate_fn, num_workers=2)

Streaming Mode Comparison


Enhancement: show streaming pipeline on WikiText-2 and compare to batch loading


In [ ]:
# ENHANCEMENT: Streaming mode — load the same dataset without downloading everything into memory
from torch.utils.data import IterableDataset

class StreamingTokenizedDataset(IterableDataset):
    def __init__(self, hf_stream, tokenizer, block_size):
        self.hf_stream  = hf_stream
        self.tokenizer  = tokenizer
        self.block_size = block_size

    def __iter__(self):
        buffer = []
        for example in self.hf_stream:
            if not example["text"].strip():
                continue
            toks = self.tokenizer(example["text"])["input_ids"]
            buffer.extend(toks)
            while len(buffer) >= self.block_size:
                chunk = buffer[:self.block_size]
                buffer = buffer[self.block_size:]
                ids = torch.tensor(chunk, dtype=torch.long)
                yield {"input_ids": ids, "labels": ids.clone()}

stream_ds   = load_dataset("wikitext", "wikitext-2-raw-v1", split="train", streaming=True)
stream_data = StreamingTokenizedDataset(stream_ds, tokenizer, block_size=128)
stream_loader = DataLoader(stream_data, batch_size=8, num_workers=0)

# Pull 3 batches to confirm it works
print("Streaming DataLoader — first 3 batch shapes:")
for i, batch in enumerate(stream_loader):
    print(f"  batch {i}: input_ids={tuple(batch['input_ids'].shape)}")
    if i == 2:
        break

print("\nKey difference vs batch loading:")
print("  Batch mode:     entire dataset downloaded + tokenized upfront, fast random access")
print("  Streaming mode: examples arrive on-the-fly, low memory, no random shuffle")

Final Sanity Check (original step 6)

In [ ]:
# 6. Final sanity check on the primary train_loader
for batch in train_loader:
    print(f"input_ids shape: {batch['input_ids'].shape}")
    print(f"labels shape:    {batch['labels'].shape}")
    print(f"Sample token IDs (first sequence): {batch['input_ids'][0][:20].tolist()}")
    break

Tokenizer Comparison Bar Chart

In [ ]:
#Tokenizer comparison bar chart — GPT-2 vs BERT vs DistilBERT
import numpy as np

sample_texts = [
    "The history of the Roman Empire spans several centuries.",
    "Tokenization is a fundamental step in NLP pipelines.",
    "Pneumonoultramicroscopicsilicovolcanoconiosis is a lung disease.",
]

short_labels = ["Sentence 1\n(Roman Empire)", "Sentence 2\n(Tokenization)", "Sentence 3\n(Long word)"]
tokenizer_names = ["GPT-2", "BERT", "DistilBERT"]
tokenizers_list = [tokenizer_gpt2, tokenizer_bert, tokenizer_distilbert]
colors = ["steelblue", "coral", "mediumseagreen"]

# token counts: shape [3 tokenizers, 3 sentences]
counts = [[len(tok.encode(s)) for s in sample_texts] for tok in tokenizers_list]

x = np.arange(len(sample_texts))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
for i, (name, color, cnt) in enumerate(zip(tokenizer_names, colors, counts)):
    bars = ax.bar(x + i * width, cnt, width, label=name, color=color, edgecolor="white")
    for bar, val in zip(bars, cnt):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                str(val), ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_xticks(x + width)
ax.set_xticklabels(short_labels, fontsize=10)
ax.set_ylabel("Number of tokens produced")
ax.set_title("Tokenizer Comparison: GPT-2 vs BERT vs DistilBERT\n(token count per sentence)", fontsize=13)
ax.legend(title="Tokenizer")
ax.set_ylim(0, max(max(c) for c in counts) + 5)
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

 Pipeline Summary Dashboard

In [ ]:
# WRAP-UP: Full pipeline summary dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Lab 1 — NLP Data Pipeline Summary", fontsize=15, fontweight="bold", y=1.01)

# ── Panel 1: Dataset overview (text stats) ──────────────────────────────────
ax1 = axes[0, 0]
stats_labels = ["Total lines", "Non-empty lines", "LM sequences\n(block=128)", "Unique tokens used"]
stats_values = [
    len(dataset),
    len(text_lengths),
    block_results[128],
    len(token_counts),
]
bar_colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
bars = ax1.barh(stats_labels, stats_values, color=bar_colors, edgecolor="white")
for bar, val in zip(bars, stats_values):
    ax1.text(bar.get_width() + max(stats_values) * 0.01, bar.get_y() + bar.get_height() / 2,
             f"{val:,}", va="center", fontsize=9)
ax1.set_title("Dataset & Pipeline Overview", fontweight="bold")
ax1.set_xlabel("Count")
ax1.xaxis.grid(True, linestyle="--", alpha=0.4)
ax1.set_axisbelow(True)
ax1.set_xlim(0, max(stats_values) * 1.18)

# ── Panel 2: Sequences produced per block size ───────────────────────────────
ax2 = axes[0, 1]
ax2.bar([str(b) for b in block_results.keys()],
        list(block_results.values()),
        color=["#4C72B0", "#55A868", "#C44E52"],
        edgecolor="white", width=0.4)
for i, (bs, cnt) in enumerate(block_results.items()):
    ax2.text(i, cnt + 100, f"{cnt:,}", ha="center", fontsize=10, fontweight="bold")
ax2.set_title("Sequences Produced by Block Size", fontweight="bold")
ax2.set_xlabel("Block size (tokens)")
ax2.set_ylabel("Number of sequences")
ax2.yaxis.grid(True, linestyle="--", alpha=0.4)
ax2.set_axisbelow(True)

# ── Panel 3: DataLoader throughput (workers benchmark) ───────────────────────
ax3 = axes[1, 0]
worker_labels = [f"num_workers={k}" for k in timing_results.keys()]
throughputs   = [50 / v for v in timing_results.values()]
bars3 = ax3.bar(worker_labels, throughputs,
                color=["#DD8452", "#4C72B0"], edgecolor="white", width=0.4)
for bar, val in zip(bars3, throughputs):
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f"{val:.1f} b/s", ha="center", fontsize=10, fontweight="bold")
ax3.set_title("DataLoader Throughput (batches/sec)", fontweight="bold")
ax3.set_ylabel("Batches per second")
ax3.yaxis.grid(True, linestyle="--", alpha=0.4)
ax3.set_axisbelow(True)
ax3.set_ylim(0, max(throughputs) * 1.25)

# ── Panel 4: Vocab coverage — tokenizer comparison ───────────────────────────
ax4 = axes[1, 1]
vocab_sizes = [tok.vocab_size for tok in tokenizers_list]
coverage    = [len(token_counts) / v * 100 for v in vocab_sizes]  # % of vocab seen in WikiText-2

x4    = np.arange(len(tokenizer_names))
bars4 = ax4.bar(tokenizer_names, vocab_sizes, color=["steelblue", "coral", "mediumseagreen"],
                edgecolor="white", width=0.4, label="Total vocab size")
ax4b  = ax4.twinx()
ax4b.plot(tokenizer_names, coverage, color="black", marker="o",
          linewidth=2, markersize=7, label="WikiText-2 coverage %")
ax4b.set_ylabel("% of vocab seen in dataset", fontsize=9)
ax4b.set_ylim(0, max(coverage) * 2)

for bar, vs in zip(bars4, vocab_sizes):
    ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
             f"{vs:,}", ha="center", fontsize=8)
for xi, cov in zip(x4, coverage):
    ax4b.text(xi, cov + 0.5, f"{cov:.1f}%", ha="center", fontsize=8, color="black")

ax4.set_title("Vocabulary Size + WikiText-2 Coverage", fontweight="bold")
ax4.set_ylabel("Total vocab size")
ax4.yaxis.grid(True, linestyle="--", alpha=0.4)
ax4.set_axisbelow(True)

fig.tight_layout()
plt.show()